# Notebook 2 — Bio-Inspired Clustering with Grey Wolf Optimization

**Workshop:** Edge Intelligence, Bio-Inspired Optimization & UAV-Assisted VANETs — ITS IRM 2026
**Based on:** Dr. Farhan Aadil's published work [J3, J4, J34]
**Estimated time:** 45–60 minutes, fully self-paced
**Prerequisite:** Notebook 1 (helpful context, not a hard dependency)

### What you'll do
VANET clustering elects a small set of **cluster heads (CHs)** so vehicles don't all broadcast
directly to infrastructure. Choosing good cluster heads is an optimization problem — you'll implement
the core update equation of **Grey Wolf Optimization (GWO)**, the same family of metaheuristic behind
[J3] and (in spirit) [J34]'s Harris Hawks approach, then benchmark it against naive random selection.

> As in Notebook 1: every claim is checked by an assertion, not by us telling you it's correct.

### Setup — run this first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_nodes(n, seed):
    """A snapshot of n vehicle positions in a 1km x 1km area."""
    rng = np.random.default_rng(seed)
    return rng.uniform(0, 1000, size=(n, 2))

def fitness(flat_centroids, nodes, k):
    """Total distance from every node to its NEAREST of the k cluster-head centroids.
    Lower = more compact clusters = less intra-cluster forwarding overhead."""
    centroids = flat_centroids.reshape(k, 2)
    d = np.linalg.norm(nodes[:, None, :] - centroids[None, :, :], axis=2)
    return d.min(axis=1).sum()

print("Setup complete.")

## Exercise 1 — Implement the Grey Wolf position-update equation

This is the actual equation from Mirjalili et al. (2014), the same one behind [J3]'s clustering result.
Three "lead wolves" (alpha, beta, delta — the current three best candidate solutions) pull every other
wolf toward them:

For each of the three leaders L ∈ {alpha, beta, delta}, with independent random numbers r1, r2 ∈ [0,1]:

```
A = 2*a*r1 - a
C = 2*r2
D = |C * X_L - X|
X_L_component = X_L - A * D
```

Then the wolf's new position is the **average of the three leader-pulled components**.

**Signature:**
```python
gwo_update_position(X, X_alpha, X_beta, X_delta, a, r) -> new_X
```
where `r` is a tuple of six numpy arrays `(r1_alpha, r2_alpha, r1_beta, r2_beta, r1_delta, r2_delta)`,
each the same shape as `X` — these are given to you (not randomly generated inside the function) so your
implementation is deterministic and testable.

In [ ]:
def gwo_update_position(X, X_alpha, X_beta, X_delta, a, r):
    r1a, r2a, r1b, r2b, r1d, r2d = r
    # YOUR CODE HERE
    # Compute the alpha-pulled component, beta-pulled component, and delta-pulled
    # component using the formula above, then return their average.
    pass

*Stuck? Expand the hint below (double-click this cell to reveal).*

<!--
HINT:
A1, C1 = 2*a*r1a - a, 2*r2a
X1 = X_alpha - A1 * np.abs(C1*X_alpha - X)

A2, C2 = 2*a*r1b - a, 2*r2b
X2 = X_beta - A2 * np.abs(C2*X_beta - X)

A3, C3 = 2*a*r1d - a, 2*r2d
X3 = X_delta - A3 * np.abs(C3*X_delta - X)

return (X1 + X2 + X3) / 3.0
-->

### Self-check — run this to verify Exercise 1

In [ ]:
_rng = np.random.default_rng(7)
_X = _rng.uniform(0, 10, size=12)
_X_alpha = _rng.uniform(0, 10, size=12)
_X_beta = _rng.uniform(0, 10, size=12)
_X_delta = _rng.uniform(0, 10, size=12)
_a = 1.3
_r = tuple(_rng.uniform(0, 1, size=12) for _ in range(6))

_r1a, _r2a, _r1b, _r2b, _r1d, _r2d = _r
_expected = ((_X_alpha - (2*_a*_r1a - _a) * np.abs(2*_r2a*_X_alpha - _X))
           + (_X_beta  - (2*_a*_r1b - _a) * np.abs(2*_r2b*_X_beta  - _X))
           + (_X_delta - (2*_a*_r1d - _a) * np.abs(2*_r2d*_X_delta - _X))) / 3.0

_got = gwo_update_position(_X, _X_alpha, _X_beta, _X_delta, _a, _r)

if _got is None:
    print("gwo_update_position returned None — did you forget a `return` statement?")
    _passed = False
else:
    _passed = np.allclose(_expected, _got, atol=1e-9)
    if not _passed:
        print("Values don't match the reference equation.")
        print("Expected:", np.round(_expected, 4))
        print("Got:     ", np.round(np.asarray(_got), 4))

if _passed:
    print("Exercise 1 self-check: ALL TESTS PASSED ✅")
assert _passed, "Exercise 1 not yet passing — see messages above."

## Exercise 2 — Run the clustering benchmark

This part is **given**. It uses your update function inside a full GWO search loop (`run_gwo`, provided
below) and compares the result against a **single naive random pick** of cluster heads — a fair,
like-for-like comparison since both approaches get to make one choice for the network snapshot.

In [ ]:
def run_gwo(nodes, k, iterations, pop_size, seed):
    rng = np.random.default_rng(seed)
    dim = k * 2
    lo, hi = 0, 1000
    wolves = rng.uniform(lo, hi, size=(pop_size, dim))
    fitnesses = np.array([fitness(w, nodes, k) for w in wolves])
    order = np.argsort(fitnesses)
    X_alpha, X_beta, X_delta = wolves[order[0]], wolves[order[1]], wolves[order[2]]
    history = [fitnesses[order[0]]]
    for it in range(iterations):
        a = 2 - it * (2 / iterations)
        for i in range(pop_size):
            r = tuple(rng.uniform(0, 1, size=dim) for _ in range(6))
            wolves[i] = np.clip(gwo_update_position(wolves[i], X_alpha, X_beta, X_delta, a, r), lo, hi)
        fitnesses = np.array([fitness(w, nodes, k) for w in wolves])
        order = np.argsort(fitnesses)
        X_alpha, X_beta, X_delta = wolves[order[0]], wolves[order[1]], wolves[order[2]]
        history.append(fitnesses[order[0]])
    return X_alpha, history

def run_single_random_pick(nodes, k, seed):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(nodes), k, replace=False)
    return fitness(nodes[idx].flatten(), nodes, k)

N_SNAPSHOTS = 15   # independent network snapshots, like 15 separate rounds
K = 6              # number of cluster heads
gwo_results, random_results = [], []

for seed in range(N_SNAPSHOTS):
    nodes = make_nodes(60, seed=200 + seed)
    _, history = run_gwo(nodes, K, iterations=40, pop_size=15, seed=seed)
    gwo_results.append(history[-1])
    random_results.append(run_single_random_pick(nodes, K, seed=seed))

print(f"GWO average intra-cluster distance:    {np.mean(gwo_results):8.1f}")
print(f"Random-pick average intra-cluster distance: {np.mean(random_results):8.1f}")

### Self-check — run this to verify your benchmark

In [ ]:
_wins = sum(g <= r for g, r in zip(gwo_results, random_results))
_improvement = 100 * (np.mean(random_results) - np.mean(gwo_results)) / np.mean(random_results)

_passed = True
if np.mean(gwo_results) >= np.mean(random_results):
    _passed = False
    print("FAILED: GWO's average fitness should be LOWER (better) than random pick's average.")
if _wins < N_SNAPSHOTS * 0.7:
    _passed = False
    print(f"FAILED: GWO only beat random pick in {_wins}/{N_SNAPSHOTS} snapshots — expected at least 70%.")

if _passed:
    print(f"Exercise 2 self-check: ALL TESTS PASSED ✅")
    print(f"GWO beat the random pick in {_wins}/{N_SNAPSHOTS} snapshots, "
          f"an average {_improvement:.1f}% reduction in intra-cluster distance "
          f"(a proxy for control/forwarding overhead).")
assert _passed

### Visualize your result

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.bar(["Random Pick", "Grey Wolf\nOptimization"],
        [np.mean(random_results), np.mean(gwo_results)],
        color=["#DCE6EC", "#065A82"])
plt.ylabel("Avg. intra-cluster distance (lower = better)")
plt.title(f"Clustering quality across {N_SNAPSHOTS} network snapshots")
plt.show()

## Reflect (no code needed)

1. GWO ran 40 iterations × 15 wolves = 600 fitness evaluations per snapshot, versus random's single pick.
   Is that a fair comparison of *solution quality*, or a fair comparison of *computational cost*? What's
   the actual trade-off a network operator faces?
2. `K = 6` cluster heads for 60 nodes. Try `K = 10` or `K = 3` — does GWO's advantage over random grow
   or shrink? Re-run the self-check; it's independent of your chosen K.
3. **Optional stretch goal (ties into Notebook 3):** Harris Hawks Optimization [J34] uses a different
   leader-following rule than GWO. If you're comfortable, try sketching a Harris-Hawks-style update
   function using the same `fitness()` — or come back to this after Notebook 3's Exercise C, where
   DeepSeek will help you design a hybrid.

---
**Next notebook:** `03_DeepSeek_Assisted_Research.ipynb`